# Week 13, Day 1: Momentum: heavy balls in long valleys

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w013/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Implement the heavy ball

Write `heavy_ball(grad, x0, lr, beta, steps)` returning the array of iterates. Check that with a constant gradient of 1 the step size settles at lr/(1 − β).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def heavy_ball(grad, x0, lr, beta, steps):
    w = np.array(x0, dtype=float)
    v = np.zeros_like(w)
    history = [w.copy()]
    for _ in range(steps):
        v = ...
        w = ...
        history.append(w.copy())
    return np.array(history)

h = heavy_ball(lambda w: np.array([1.0]), [0.0], 0.1, 0.9, 60)
plt.figure(figsize=(5, 2.8)); plt.plot(-np.diff(h[:, 0])); plt.axhline(1.0, ls="--", color="0.5")
plt.xlabel("step"); plt.ylabel("step length"); plt.show()

In [ ]:
import numpy as np
def test_zero_beta():
    "with beta 0 the method is gradient descent"
    g = lambda w: 2 * w
    h = heavy_ball(g, np.array([1.0]), 0.1, 0.0, 3)
    assert np.allclose(h[:, 0], [1, 0.8, 0.64, 0.512])

def test_velocity():
    "with a constant gradient the step approaches lr / (1 - beta)"
    h = heavy_ball(lambda w: np.array([1.0]), np.array([0.0]), 0.1, 0.9, 200)
    assert np.isclose(h[-1, 0] - h[-2, 0], -0.1 / (1 - 0.9), rtol=1e-3)

test_zero_beta()
test_velocity()
print("All checks passed")

### Exercise: Gradient descent against the heavy ball

For Hessians diag(κ, 1) with κ = 2, 10 and 100, count the steps until the distance from the minimum (starting at (1, 1)) falls below 1e-6, for gradient descent with $\eta = 2/(\kappa + 1)$ and for the heavy ball with the optimal η* and β* above. Store the counts in `steps_gd` and `steps_hb`.

In [ ]:
def steps_to(history, tol=1e-6):
    d = np.linalg.norm(history, axis=1)
    return int(np.argmax(d < tol)) if np.any(d < tol) else len(d)

kappas = [2.0, 10.0, 100.0]
steps_gd, steps_hb = [], []
for kap in kappas:
    H = np.diag([kap, 1.0]); g = lambda w, H=H: H @ w
    steps_gd.append(steps_to(heavy_ball(g, [1.0, 1.0], 2 / (kap + 1), 0.0, 5000)))
    lr_star = ...
    beta_star = ...
    steps_hb.append(steps_to(heavy_ball(g, [1.0, 1.0], lr_star, beta_star, 5000)))
print("kappa:", kappas, "\ngradient descent:", steps_gd, "\nheavy ball:      ", steps_hb)

In [ ]:
import numpy as np
def test_counts():
    "steps holds the number of steps to a millionfold reduction for GD and the heavy ball, for each kappa"
    assert len(steps_gd) == len(kappas) == len(steps_hb)

def test_speedup():
    "for kappa = 100 the heavy ball needs fewer than a fifth of gradient descent's steps"
    assert steps_hb[-1] < steps_gd[-1] / 5

def test_sqrt():
    "the heavy ball's steps grow roughly like sqrt(kappa): from kappa 10 to 100 by a factor between 2 and 5"
    assert 2 < steps_hb[-1] / steps_hb[1] < 5

test_counts()
test_speedup()
test_sqrt()
print("All checks passed")

In [ ]:
rng = np.random.default_rng(2018)
N, T = 50, 1500
prefs = np.linspace(0, np.pi, N, endpoint=False)
theta = rng.uniform(0, np.pi, T)
labels = (np.cos(2 * (theta - np.pi / 4)) > 0).astype(float)
counts = rng.poisson((10 + 20 * np.exp(2 * (np.cos(2 * (theta[:, None] - prefs[None])) - 1))) * 0.2)
Z = np.column_stack([counts, np.ones(T)])          # raw counts: badly conditioned (Week 12 Forge)
def sigmoid(u): return 1 / (1 + np.exp(-u))
def xent(w): p = np.clip(sigmoid(Z @ w), 1e-12, 1 - 1e-12); return -np.mean(labels * np.log(p) + (1 - labels) * np.log(1 - p))
def grad(w): return Z.T @ (sigmoid(Z @ w) - labels) / T
lam_max = np.linalg.eigvalsh(Z.T @ Z / (4 * T)).max()
print("largest curvature", round(lam_max, 1))

### Exercise: Momentum on a badly conditioned decoder

Train the logistic decoder on raw spike counts for 300 steps with gradient descent at learning rate 1.9/λmax, and with the heavy ball at the same learning rate with β = 0.9. Record the loss at every iterate in `loss_gd` and `loss_mom`.

In [ ]:
lr = 1.9 / lam_max
h_gd = heavy_ball(grad, np.zeros(N + 1), lr, 0.0, 300)
h_mom = ...
loss_gd = [xent(w) for w in h_gd]
loss_mom = [xent(w) for w in h_mom]

plt.figure(figsize=(5, 3)); plt.plot(loss_gd, label="gradient descent"); plt.plot(loss_mom, label="momentum 0.9")
plt.xlabel("step"); plt.ylabel("training loss"); plt.legend(frameon=False); plt.show()
print(f"after 300 steps: GD {loss_gd[-1]:.4f}, momentum {loss_mom[-1]:.4f}")

In [ ]:
import numpy as np
def test_runs():
    "both runs record 300 losses"
    assert len(loss_gd) == len(loss_mom) == 301

def test_better():
    "momentum reaches a clearly lower loss than gradient descent in 300 steps"
    assert loss_mom[-1] < loss_gd[-1] - 0.01

test_runs()
test_better()
print("All checks passed")